# Research Graph Tools: a gradual walkthrough

Start with the first cell and run downward. Each section adds one idea, from making a graph to exploring and visualizing results. From the repository directory, run `uv sync` and use that project environment as this notebook's kernel. The first code cell also points Python at this checkout's `src` folder.

## 1. Create a graph

A graph is a set of nodes connected by edges. These helpers create common examples: a path is a line; a cycle loops back to its start.

In [ ]:
# Make the package importable when this notebook is run from the repository checkout.
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "research_graph_tools").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from research_graph_tools.graphs import path_graph, cycle_graph

path = path_graph(4)
cycle = cycle_graph(4)

## 2. Look at its parts

Nodes can be numbers, text, or other hashable values. NetworkX supplies the basic graph operations.

In [ ]:
print("Nodes:", list(path.nodes))
print("Edges:", list(path.edges))
print("Number of nodes:", path.number_of_nodes())
print("Number of edges:", path.number_of_edges())

## 3. Compare two simple graphs

A path has two ends; a cycle connects the ends together. Both constructors return ordinary NetworkX graphs.

In [ ]:
print("Path edges:", list(path.edges))
print("Cycle edges:", list(cycle.edges))

## 4. Calculate one number

Kemeny's constant summarizes expected travel time for a simple random walk on a connected graph. The function returns a single number.

In [ ]:
from research_graph_tools.metrics import kemeny_constant

k = kemeny_constant(path)
print("Kemeny's constant:", k)

## 5. Get results for each node

Some metrics return dictionaries keyed by node. Accessibility gives one value per node; the Fiedler vector does too. These values can help compare positions in the graph.

In [ ]:
from research_graph_tools.metrics import accessibility_index, fiedler_vector

accessibility = accessibility_index(path)
fiedler = fiedler_vector(path)
print("Accessibility:", accessibility)
print("Fiedler vector:", fiedler)

## 6. Calculate an approximate conductance

The Cheeger helper returns one value, using a sweep over the normalized Fiedler vector to choose a cut.

In [ ]:
from research_graph_tools.metrics import cheeger_approximation

print("Approximate conductance:", cheeger_approximation(path))

## 7. Draw and compare graphs

`draw_graph(graph)` shows one graph in the Counterexample Atlas style. Pass an added edge to show baseline and augmented panels; add `show_delta=True` for a third panel. Atlas graph records can use their default edge with `show_augmented=True` or `show_delta=True`.

In [ ]:
from IPython.display import HTML, display
from research_graph_tools.visualization import draw_graph

display(HTML(draw_graph(path).to_html()))
display(HTML(draw_graph(path, (0, 3), show_delta=True).to_html()))

## 8. Search candidates

`find_counterexample` checks candidates in order and returns the first one that passes your condition. Here we search paths for one whose Kemeny constant exceeds 5. `limit` caps the number checked.

In [ ]:
from research_graph_tools.search import find_counterexample

candidates = [path_graph(n) for n in range(3, 10)]
found = find_counterexample(
    candidates,
    lambda graph: kemeny_constant(graph) > 5,
    limit=7,
)
print("Found a candidate:", found is not None)
if found is not None:
    print("Node count:", found.number_of_nodes())
    print("Kemeny's constant:", kemeny_constant(found))

## 9. Use your own graph

You can also create a graph with NetworkX and pass it to the package tools. Metrics require a connected, undirected graph.

In [ ]:
import networkx as nx

custom = nx.Graph()
custom.add_edges_from([("A", "B"), ("B", "C"), ("C", "A"), ("C", "D")])
print("Kemeny's constant:", kemeny_constant(custom))
display(HTML(draw_graph(custom, color_metric="fiedler").to_html()))

## 10. Visualize a graph from an adjacency matrix

`draw_graph` accepts a symmetric 0/1 NumPy adjacency matrix directly. It assigns numbered nodes and chooses a layout automatically. Positions are optional.

In [ ]:
import numpy as np

A = np.array([
    [0, 1, 0, 1],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 0, 1, 0],
])

display(HTML(draw_graph(A).to_html()))
display(HTML(draw_graph(A, (0, 2), show_delta=True).to_html()))